In [2]:
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, Subset, DataLoader
import numpy as np

from dataset import create_dataloaders
from tcn_model import MEGTCN
from gan_model import MEGGAN
from mlp_model import MLP
from cnn_baseline_1d import CNNBaseline1D
from train import train_one_epoch
from evaluate import evaluate, evaluate_top_models_person_cv
from grid_search import run_grid_search, run_person_grid_search

In [4]:
BATCH_SIZE = 8

In [5]:
INTRA_DATA_DIR = "preprocessed_data/Intra"
intra_train_loader, intra_test_loader = create_dataloaders(INTRA_DATA_DIR, BATCH_SIZE, add_person_id=True)

Loading test data...
Loading train data...
Loaded 32 training samples
Loaded 8 test samples
Class distribution in training: [8 8 8 8]
Class distribution in test: [2 2 2 2]
Train batches: 4
Test batches: 1


### Cross Dataset

In [6]:
CROSS_DATA_DIR = "preprocessed_data/Cross"
BATCH_SIZE = 8
EPOCHS = 30
LEARNING_RATE = 1e-3
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

cross_train_loader, cross_test_loader = create_dataloaders(CROSS_DATA_DIR, BATCH_SIZE, add_person_id=True)

Using device: cuda
Loading test1 data...
Loading test2 data...
Loading test3 data...
Loading train data...
Loaded 64 training samples
Loaded 48 test samples
Class distribution in training: [16 16 16 16]
Class distribution in test: [12 12 12 12]
Train batches: 8
Test batches: 6


In [7]:
cross_dataset = cross_train_loader.dataset
intra_dataset = intra_train_loader.dataset

def build_person_subsets(dataset, source_name):
    """Build subsets of the dataset grouped by person_id."""
    if dataset.person_ids is None:
        raise ValueError(f"{source_name} dataset must be created with add_person_id=True")

    grouped_indices = {}
    for index, person_id in enumerate(dataset.person_ids):
        grouped_indices.setdefault(int(person_id), []).append(index)

    return [
        (f"{source_name}_{person_id}", Subset(dataset, indices), person_id)
        for person_id, indices in sorted(grouped_indices.items())
    ]


cross_person_splits = build_person_subsets(cross_dataset, "cross")
intra_person_splits = build_person_subsets(intra_dataset, "intra")
person_splits = cross_person_splits + intra_person_splits

print("Person-level folds:")
for split_name, subset, person_id in person_splits:
    print(f"  {split_name}: person_id={person_id}, samples={len(subset)}")

Person-level folds:
  cross_113922: person_id=113922, samples=32
  cross_164636: person_id=164636, samples=32
  intra_105923: person_id=105923, samples=32


In [8]:
def cross_validation(
    model_class,
    person_splits,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    model_kwargs=None,
    weight_decay=1e-4,
    patience=8,
    min_delta=1e-3,
):
    """Train one model per person split with early stopping.

    The key anti-overfitting pieces here are:
    - weight decay in the optimizer
    - learning-rate reduction when validation loss plateaus
    - early stopping that restores the best validation checkpoint
    """

    fold_accuracies = []
    model_kwargs = model_kwargs or {}
    use_pin_memory = DEVICE == "cuda"

    for fold, (test_name, test_subset, test_person_id) in enumerate(person_splits):
        train_subsets = [
            subset
            for split_name, subset, _ in person_splits
            if split_name != test_name
        ]

        fold_train_dataset = ConcatDataset(train_subsets)
        fold_train_loader = DataLoader(
            fold_train_dataset,
            batch_size=batch_size,
            shuffle=True,
            pin_memory=use_pin_memory,
        )
        val_loader = DataLoader(
            test_subset,
            batch_size=batch_size,
            shuffle=False,
            pin_memory=use_pin_memory,
        )

        model = model_class(num_classes=NUM_CLASSES, **model_kwargs).to(DEVICE)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            patience=max(1, patience // 2),
            factor=0.5,
        )

        print(
            f"Starting fold {fold + 1}/{len(person_splits)} | "
            f"Test person: {test_name} (person_id={test_person_id})"
        )

        best_val_loss = float("inf")
        best_val_acc = 0.0
        best_state_dict = None
        bad_epochs = 0

        for epoch in range(epochs):
            train_loss, train_acc = train_one_epoch(
                model,
                fold_train_loader,
                criterion,
                optimizer,
                DEVICE,
            )
            val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
            scheduler.step(val_loss)

            # Keep the checkpoint with the lowest validation loss.
            if val_loss < best_val_loss - min_delta:
                best_val_loss = val_loss
                best_val_acc = val_acc
                best_state_dict = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }
                bad_epochs = 0
            else:
                bad_epochs += 1

            print(
                f"Fold {fold + 1} | Epoch {epoch + 1}/{epochs} | "
                f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | "
                f"Val Loss: {val_loss:.4f}"
            )

            # Stop once validation loss has not improved for several epochs.
            if bad_epochs >= patience:
                print(
                    f"Early stopping at epoch {epoch + 1} after {patience} "
                    "non-improving epochs."
                )
                break

        # Restore the best model before recording the fold result.
        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)

        fold_accuracies.append(best_val_acc)

    print(
        f"Mean person-level validation accuracy: {np.mean(fold_accuracies):.2f}%"
    )

In [26]:
MEGGAN_FAST_CONFIG = {
    "temporal_hidden": 32,
    "graph_hidden": 64,
    "dropout": 0.25,
    #"attention_dropout": 0.2,
    #"num_neighbors": 6,
}

MEGTCN_CONFIG = {
    "kernel_size": 3,
    "dropout": 0.2,
    "hidden_channels": 32,
}

# cross_validation(
#     MEGTCN,
#     person_splits,
#     batch_size=16,
#     epochs=100,
#     model_kwargs=MEGTCN_CONFIG,
#     weight_decay=1e-4,
#     patience=16,
# )

cross_validation(
    MEGGAN,
    person_splits,
    batch_size=16,
    epochs=100,
    model_kwargs=MEGGAN_FAST_CONFIG,
    weight_decay=5e-4,
    patience=16,
)

Starting fold 1/3 | Test person: cross_113922 (person_id=113922)
Fold 1 | Epoch 1/100 | Train Acc: 26.56% | Val Acc: 40.62% | Val Loss: 1.3759
Fold 1 | Epoch 2/100 | Train Acc: 37.50% | Val Acc: 50.00% | Val Loss: 1.3419
Fold 1 | Epoch 3/100 | Train Acc: 35.94% | Val Acc: 34.38% | Val Loss: 1.3089
Fold 1 | Epoch 4/100 | Train Acc: 40.62% | Val Acc: 31.25% | Val Loss: 1.2644
Fold 1 | Epoch 5/100 | Train Acc: 43.75% | Val Acc: 46.88% | Val Loss: 1.2334
Fold 1 | Epoch 6/100 | Train Acc: 40.62% | Val Acc: 28.12% | Val Loss: 1.2212
Fold 1 | Epoch 7/100 | Train Acc: 56.25% | Val Acc: 34.38% | Val Loss: 1.2106
Fold 1 | Epoch 8/100 | Train Acc: 43.75% | Val Acc: 37.50% | Val Loss: 1.2001
Fold 1 | Epoch 9/100 | Train Acc: 51.56% | Val Acc: 37.50% | Val Loss: 1.2155
Fold 1 | Epoch 10/100 | Train Acc: 54.69% | Val Acc: 37.50% | Val Loss: 1.2245
Fold 1 | Epoch 11/100 | Train Acc: 62.50% | Val Acc: 37.50% | Val Loss: 1.2123
Fold 1 | Epoch 12/100 | Train Acc: 60.94% | Val Acc: 31.25% | Val Loss: 1.2

# Grid Search, using person splits

In [9]:
cross_person_splits = build_person_subsets(cross_dataset, "cross")
intra_person_splits = build_person_subsets(intra_dataset, "intra")
person_splits = cross_person_splits + intra_person_splits

print("Person-level folds:")
for split_name, subset, person_id in person_splits:
    print(f"  {split_name}: person_id={person_id}, samples={len(subset)}")

Person-level folds:
  cross_113922: person_id=113922, samples=32
  cross_164636: person_id=164636, samples=32
  intra_105923: person_id=105923, samples=32


# TCN MODEL

In [ ]:
from tcn_model import MEGTCN

param_grid = {
    "learning_rate": [1e-3, 5e-3, 5e-4, 1e-4],
    "kernel_size": [5, 7, 11],
    "dropout": [0.05, 0.1, 0.2, 0.5],
    "hidden_channels": [16, 32, 64],
    "batch_size": [8, 16, 32, 64],
}

top_models = run_person_grid_search(
    model_class=MEGTCN,
    param_grid=param_grid,
    person_splits=person_splits,
    num_classes=4,
    epochs=100
)

Total configs: 144

Testing parameters:
{'learning_rate': 0.005, 'kernel_size': 5, 'dropout': 0.05, 'hidden_channels': 16, 'batch_size': 8}

Fold 1/3 | Test person: 113922
Fold 1 | Epoch 1/100 | Train Acc: 65.62% | Val Acc: 75.00% | Val Loss: 0.9204
Fold 1 | Epoch 2/100 | Train Acc: 85.94% | Val Acc: 56.25% | Val Loss: 1.1517
Fold 1 | Epoch 3/100 | Train Acc: 90.62% | Val Acc: 78.12% | Val Loss: 0.8830
Fold 1 | Epoch 4/100 | Train Acc: 98.44% | Val Acc: 65.62% | Val Loss: 1.2711
Fold 1 | Epoch 5/100 | Train Acc: 95.31% | Val Acc: 65.62% | Val Loss: 1.5847
Fold 1 | Epoch 6/100 | Train Acc: 100.00% | Val Acc: 62.50% | Val Loss: 1.5618
Fold 1 | Epoch 7/100 | Train Acc: 98.44% | Val Acc: 59.38% | Val Loss: 1.5392
Fold 1 | Epoch 8/100 | Train Acc: 100.00% | Val Acc: 59.38% | Val Loss: 1.5767
Fold 1 | Epoch 9/100 | Train Acc: 100.00% | Val Acc: 53.12% | Val Loss: 1.7620
Fold 1 | Epoch 10/100 | Train Acc: 100.00% | Val Acc: 56.25% | Val Loss: 1.7855
Fold 1 | Epoch 11/100 | Train Acc: 100.00% 

In [9]:
results = evaluate_top_models_person_cv(
    model_class=MEGTCN,
    top_models=top_models,
    person_splits=person_splits,
    num_classes=4,
    device=DEVICE,
    n_runs=25,
)


MODEL 1
{'learning_rate': 0.005, 'kernel_size': 5, 'dropout': 0.1, 'hidden_channels': 16, 'batch_size': 32}

Run 1/25
  Fold 1/3 | Person 113922 | Acc 71.88%
  Fold 2/3 | Person 164636 | Acc 59.38%
  Fold 3/3 | Person 105923 | Acc 78.12%
Run 1 Mean Person-CV Accuracy: 69.79%

Run 2/25
  Fold 1/3 | Person 113922 | Acc 46.88%
  Fold 2/3 | Person 164636 | Acc 56.25%
  Fold 3/3 | Person 105923 | Acc 75.00%
Run 2 Mean Person-CV Accuracy: 59.38%

Run 3/25
  Fold 1/3 | Person 113922 | Acc 71.88%
  Fold 2/3 | Person 164636 | Acc 59.38%
  Fold 3/3 | Person 105923 | Acc 78.12%
Run 3 Mean Person-CV Accuracy: 69.79%

Run 4/25
  Fold 1/3 | Person 113922 | Acc 71.88%
  Fold 2/3 | Person 164636 | Acc 71.88%
  Fold 3/3 | Person 105923 | Acc 68.75%
Run 4 Mean Person-CV Accuracy: 70.83%

Run 5/25
  Fold 1/3 | Person 113922 | Acc 65.62%
  Fold 2/3 | Person 164636 | Acc 62.50%
  Fold 3/3 | Person 105923 | Acc 96.88%
Run 5 Mean Person-CV Accuracy: 75.00%

Run 6/25
  Fold 1/3 | Person 113922 | Acc 71.88%
 

# GAN MODEL

In [12]:
# from gan_model_simple import MEGGAN

param_grid = {
    "learning_rate": [5e-3],
    "hidden_channels": [32],
    "kernel_size": [3],
    "dropout": [0.1],
    "num_heads": [2],
    "weight_decay": [1e-4],
    "batch_size": [8],
}

top_models = run_person_grid_search(
    model_class=MEGGAN,
    param_grid=param_grid,
    person_splits=person_splits,
    num_classes=4,
    epochs=100,
    patience=20
)

Total configs: 1

Testing parameters:
{'learning_rate': 0.005, 'hidden_channels': 32, 'kernel_size': 3, 'dropout': 0.1, 'num_heads': 2, 'weight_decay': 0.0001, 'batch_size': 8}

Fold 1/3 | Test person: 113922
Fold 1 | Epoch 1/100 | Train Acc: 76.56% | Val Acc: 71.88% | Val Loss: 1.0330
Fold 1 | Epoch 2/100 | Train Acc: 90.62% | Val Acc: 65.62% | Val Loss: 1.0388
Fold 1 | Epoch 3/100 | Train Acc: 96.88% | Val Acc: 68.75% | Val Loss: 0.8714
Fold 1 | Epoch 4/100 | Train Acc: 100.00% | Val Acc: 75.00% | Val Loss: 0.8274
Fold 1 | Epoch 5/100 | Train Acc: 95.31% | Val Acc: 62.50% | Val Loss: 0.9464
Fold 1 | Epoch 6/100 | Train Acc: 95.31% | Val Acc: 75.00% | Val Loss: 0.7495
Fold 1 | Epoch 7/100 | Train Acc: 100.00% | Val Acc: 81.25% | Val Loss: 0.5769
Fold 1 | Epoch 8/100 | Train Acc: 100.00% | Val Acc: 75.00% | Val Loss: 0.5855
Fold 1 | Epoch 9/100 | Train Acc: 95.31% | Val Acc: 75.00% | Val Loss: 0.6641
Fold 1 | Epoch 10/100 | Train Acc: 100.00% | Val Acc: 75.00% | Val Loss: 0.6447
Fold 1

In [13]:
results = evaluate_top_models_person_cv(
    model_class=MEGGAN,
    top_models=top_models,
    person_splits=person_splits,
    num_classes=4,
    device=DEVICE,
    n_runs=25,
)


MODEL 1
{'learning_rate': 0.005, 'hidden_channels': 32, 'kernel_size': 3, 'dropout': 0.1, 'num_heads': 2, 'weight_decay': 0.0001, 'batch_size': 8}

Run 1/25
  Fold 1/3 | Person 113922 | Acc 75.00%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 65.62%
Run 1 Mean Person-CV Accuracy: 71.88%

Run 2/25
  Fold 1/3 | Person 113922 | Acc 78.12%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 90.62%
Run 2 Mean Person-CV Accuracy: 81.25%

Run 3/25
  Fold 1/3 | Person 113922 | Acc 62.50%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 93.75%
Run 3 Mean Person-CV Accuracy: 77.08%

Run 4/25
  Fold 1/3 | Person 113922 | Acc 81.25%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 87.50%
Run 4 Mean Person-CV Accuracy: 81.25%

Run 5/25
  Fold 1/3 | Person 113922 | Acc 62.50%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 100.00%
Run 5 Mean Person-CV Accuracy: 79.17%

Run 6/25
 

# CNN MODEL

In [9]:
param_grid = {
    "learning_rate": [1e-3, 5e-3, 5e-4, 1e-4],
    "kernel_size": [5, 7, 11],
    "dropout": [0.05, 0.1, 0.2, 0.5],
    "hidden_channels": [16, 32, 64],
    "batch_size": [8, 16, 32, 64],
}

top_models = run_person_grid_search(
    model_class=CNNBaseline1D,
    param_grid=param_grid,
    person_splits=person_splits,
    num_classes=4,
    epochs=100
)

Total configs: 576

Testing parameters:
{'learning_rate': 0.001, 'kernel_size': 5, 'dropout': 0.05, 'hidden_channels': 16, 'batch_size': 8}

Fold 1/3 | Test person: 113922
Fold 1 | Epoch 1/100 | Train Acc: 53.12% | Val Acc: 56.25% | Val Loss: 1.2422
Fold 1 | Epoch 2/100 | Train Acc: 79.69% | Val Acc: 65.62% | Val Loss: 1.2186
Fold 1 | Epoch 3/100 | Train Acc: 79.69% | Val Acc: 65.62% | Val Loss: 1.1084
Fold 1 | Epoch 4/100 | Train Acc: 92.19% | Val Acc: 68.75% | Val Loss: 1.0420
Fold 1 | Epoch 5/100 | Train Acc: 92.19% | Val Acc: 68.75% | Val Loss: 0.9979
Fold 1 | Epoch 6/100 | Train Acc: 98.44% | Val Acc: 75.00% | Val Loss: 0.9400
Fold 1 | Epoch 7/100 | Train Acc: 100.00% | Val Acc: 68.75% | Val Loss: 0.8894
Fold 1 | Epoch 8/100 | Train Acc: 96.88% | Val Acc: 75.00% | Val Loss: 0.8632
Fold 1 | Epoch 9/100 | Train Acc: 96.88% | Val Acc: 71.88% | Val Loss: 0.8243
Fold 1 | Epoch 10/100 | Train Acc: 95.31% | Val Acc: 75.00% | Val Loss: 0.7762
Fold 1 | Epoch 11/100 | Train Acc: 100.00% | V

In [10]:
results = evaluate_top_models_person_cv(
    model_class=CNNBaseline1D,
    top_models=top_models,
    person_splits=person_splits,
    num_classes=4,
    device=DEVICE,
    n_runs=25,
)


MODEL 1
{'learning_rate': 0.001, 'kernel_size': 5, 'dropout': 0.05, 'hidden_channels': 16, 'batch_size': 8}

Run 1/25
  Fold 1/3 | Person 113922 | Acc 87.50%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 96.88%
Run 1 Mean Person-CV Accuracy: 86.46%

Run 2/25
  Fold 1/3 | Person 113922 | Acc 78.12%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 100.00%
Run 2 Mean Person-CV Accuracy: 84.38%

Run 3/25
  Fold 1/3 | Person 113922 | Acc 78.12%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 93.75%
Run 3 Mean Person-CV Accuracy: 82.29%

Run 4/25
  Fold 1/3 | Person 113922 | Acc 75.00%
  Fold 2/3 | Person 164636 | Acc 50.00%
  Fold 3/3 | Person 105923 | Acc 96.88%
Run 4 Mean Person-CV Accuracy: 73.96%

Run 5/25
  Fold 1/3 | Person 113922 | Acc 75.00%
  Fold 2/3 | Person 164636 | Acc 75.00%
  Fold 3/3 | Person 105923 | Acc 96.88%
Run 5 Mean Person-CV Accuracy: 82.29%

Run 6/25
  Fold 1/3 | Person 113922 | Acc 87.50%


# MLP MODEL

In [12]:
param_grid = {
    "learning_rate": [1e-3, 5e-3, 5e-4, 1e-4],
    "dropout": [0.05, 0.1, 0.2, 0.5],
    "hidden_size": [16, 32, 64],
    "batch_size": [8, 16, 32, 64],
}

top_models = run_person_grid_search(
    model_class=MLP,
    param_grid=param_grid,
    person_splits=person_splits,
    num_classes=4,
    epochs=100
)

Total configs: 192

Testing parameters:
{'learning_rate': 0.001, 'dropout': 0.05, 'hidden_size': 16, 'batch_size': 8}

Fold 1/3 | Test person: 113922
Fold 1 | Epoch 1/100 | Train Acc: 59.38% | Val Acc: 62.50% | Val Loss: 22.4114
Fold 1 | Epoch 2/100 | Train Acc: 71.88% | Val Acc: 50.00% | Val Loss: 44.8741
Fold 1 | Epoch 3/100 | Train Acc: 82.81% | Val Acc: 56.25% | Val Loss: 30.1027
Fold 1 | Epoch 4/100 | Train Acc: 98.44% | Val Acc: 50.00% | Val Loss: 38.8843
Fold 1 | Epoch 5/100 | Train Acc: 95.31% | Val Acc: 46.88% | Val Loss: 61.3062
Fold 1 | Epoch 6/100 | Train Acc: 93.75% | Val Acc: 46.88% | Val Loss: 66.3728
Fold 1 | Epoch 7/100 | Train Acc: 96.88% | Val Acc: 62.50% | Val Loss: 70.0223
Fold 1 | Epoch 8/100 | Train Acc: 98.44% | Val Acc: 59.38% | Val Loss: 70.8860
Fold 1 | Epoch 9/100 | Train Acc: 95.31% | Val Acc: 62.50% | Val Loss: 79.8896
Early stopping at epoch 9 after 8 non-improving epochs.

Fold 2/3 | Test person: 164636
Fold 2 | Epoch 1/100 | Train Acc: 64.06% | Val Acc:

In [13]:
results = evaluate_top_models_person_cv(
    model_class=MLP,
    top_models=top_models,
    person_splits=person_splits,
    num_classes=4,
    device=DEVICE,
    n_runs=25,
)


MODEL 1
{'learning_rate': 0.001, 'dropout': 0.05, 'hidden_size': 16, 'batch_size': 8}

Run 1/25
  Fold 1/3 | Person 113922 | Acc 56.25%
  Fold 2/3 | Person 164636 | Acc 37.50%
  Fold 3/3 | Person 105923 | Acc 46.88%
Run 1 Mean Person-CV Accuracy: 46.88%

Run 2/25
  Fold 1/3 | Person 113922 | Acc 28.12%
  Fold 2/3 | Person 164636 | Acc 46.88%
  Fold 3/3 | Person 105923 | Acc 46.88%
Run 2 Mean Person-CV Accuracy: 40.62%

Run 3/25
  Fold 1/3 | Person 113922 | Acc 62.50%
  Fold 2/3 | Person 164636 | Acc 46.88%
  Fold 3/3 | Person 105923 | Acc 56.25%
Run 3 Mean Person-CV Accuracy: 55.21%

Run 4/25
  Fold 1/3 | Person 113922 | Acc 59.38%
  Fold 2/3 | Person 164636 | Acc 81.25%
  Fold 3/3 | Person 105923 | Acc 46.88%
Run 4 Mean Person-CV Accuracy: 62.50%

Run 5/25
  Fold 1/3 | Person 113922 | Acc 56.25%
  Fold 2/3 | Person 164636 | Acc 46.88%
  Fold 3/3 | Person 105923 | Acc 31.25%
Run 5 Mean Person-CV Accuracy: 44.79%

Run 6/25
  Fold 1/3 | Person 113922 | Acc 59.38%
  Fold 2/3 | Person 164